# 流式响应与 SSE

学习目标：发送有限的分段响应和 SSE 事件，通过真实客户端与浏览器逐步消费，并确认正常结束及提前断开后的清理。

前置知识：HTTP 响应、异步生成器、任务取消与 JavaScript 基础。

适用条件：Python 3.12、FastAPI 0.141.1、HTTPX 0.28.1。原生 EventSourceResponse 与 ServerSentEvent 从 FastAPI 0.135.0 引入，本章使用 fastapi.sse，不需要额外 SSE 库。

环境准备：[FastAPI 环境与运行说明](README.md)。选择课程环境的 Python 3 (ipykernel)，从空内核顺序执行。Notebook 和终端工作目录均为 content/Web与应用开发/FastAPI。第 2 节起需要运行本地端口 8200，浏览器与客户端实验结束后关闭服务。

配套脚本：位于 scripts/20-streaming-and-sse/。

1. [app.py](scripts/20-streaming-and-sse/app.py)：组合本章的文本流、SSE、状态与页面路由，供 Uvicorn 导入。
2. [events.html](scripts/20-streaming-and-sse/events.html)：原生 EventSource 页面，可正常接收或在首个事件后关闭。

## 1 用有限生成器逐段提供数据

流式响应把生成器提供的数据逐步写入响应体。先观察一个只生成三段字节的异步生成器；yield 交出本段，下一次迭代再继续执行。

await 给事件循环处理其他任务与取消的机会，finally 在生成器退出时执行。这里直接迭代只观察 Python 生成器，尚未发送 HTTP 响应。

In [1]:
import asyncio
from collections.abc import AsyncIterator

async def text_chunks() -> AsyncIterator[bytes]:
    try:
        for number in range(1, 4):
            yield f"part {number}\n".encode("utf-8")
            if number < 3:
                await asyncio.sleep(0.1)
    finally:
        print("文本生成器已退出", flush=True)

generated = [part async for part in text_chunks()]
assert generated == [b"part 1\n", b"part 2\n", b"part 3\n"]
print(generated)

文本生成器已退出


[b'part 1\n', b'part 2\n', b'part 3\n']


## 2 用 StreamingResponse 发送真实响应

StreamingResponse 接收普通或异步生成器，下面指定 text/plain 作为媒体类型。这个 Code 单元只定义 app，不启动端口服务。

In [2]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI(title="有限流式响应")

@app.get("/stream")
def stream_text() -> StreamingResponse:
    return StreamingResponse(text_chunks(), media_type="text/plain")

真实网络可能改变数据被客户端接收时的分块。TCP 提供字节流，不保证网络分段与服务端每次 yield 一一对应；HTTPX 还可以按 chunk_size 再组织读取结果。要识别完整文本行或 SSE 事件，应按相应格式解析。

配套 app.py 包含本节及后续讲解的路由，导入模块不会自行启动服务器。

Step 1：在课程目录的终端启动本地服务。

```powershell
python -m uvicorn app:app --app-dir scripts/20-streaming-and-sse --host 127.0.0.1 --port 8200
```

Step 2：等待终端显示应用启动完成，再执行下面及后续代码单元。

HTTPX 的 stream 上下文在收到响应头后交给调用方逐步读取，退出时关闭响应。这里故意让客户端每次输出最多 4 字节；这些是客户端整理的字节块，不是抓取的 TCP 数据包。trust_env=False 让本地请求不使用环境代理配置。

In [3]:
import httpx

base_url = "http://127.0.0.1:8200"
async with httpx.AsyncClient(base_url=base_url, timeout=3.0, trust_env=False) as client:
    async with client.stream("GET", "/stream") as response:
        response.raise_for_status()
        received = []
        async for part in response.aiter_bytes(chunk_size=4):
            received.append(part)
            print(repr(part))

# 检查拼接后的内容，不要求读取次数等于服务端的三次 yield。
assert b"".join(received) == b"part 1\npart 2\npart 3\n"
assert response.is_closed and client.is_closed
print("响应与客户端已关闭：", response.is_closed, client.is_closed)

b'part'
b' 1\np'
b'art '


b'2\npa'
b'rt 3'
b'\n'
响应与客户端已关闭： True True


## 3 按文本行消费

aiter_lines 在客户端处理文本解码和行分隔，适合按行约定的文本流。它返回的每行不包含行尾换行符，不需要假设网络字节块刚好是一整行。

先拿到第一行，再继续消费剩余行。这次仍通过实际端口读取，不使用可能先收集完整响应的应用内传输结果来证明逐步到达。

In [4]:
async with httpx.AsyncClient(base_url=base_url, timeout=3.0, trust_env=False) as client:
    async with client.stream("GET", "/stream") as response:
        response.raise_for_status()
        lines = response.aiter_lines()
        first = await anext(lines)
        print("已读到第一行：", first)
        rest = [line async for line in lines]

assert first == "part 1" and rest == ["part 2", "part 3"]
assert response.is_closed and client.is_closed
print("其余行：", rest)

已读到第一行： part 1


其余行： ['part 2', 'part 3']


## 4 用原生 SSE 表示事件

服务器发送事件（Server-Sent Events，SSE）是在 HTTP 上由服务端向客户端发送文本事件的格式，媒体类型为 text/event-stream，使用 UTF-8。一个事件由字段行组成，空行结束当前事件；它比任意文本块多了一层事件边界。

| 字段 | 中文名称／含义 | 本章示例 |
| --- | --- | --- |
| data | 事件数据 | 包含 step 和 text 的 JSON |
| event | 事件类型；省略时浏览器使用 message | update 或 done |
| id | 事件标识；浏览器可在重连时通过 Last-Event-ID 发回 | 字符串 1、2、3 |
| retry | 浏览器重连等待时间，单位为毫秒 | 1000 |

FastAPI 的 ServerSentEvent 会把 data 编码为 JSON，字符串也会按 JSON 字符串编码。不要事先把字典转成 JSON 字符串再传给 data；浏览器随后对 event.data 使用 JSON.parse。id 本身不实现历史事件重放，服务端若要续传还需处理 Last-Event-ID。

In [5]:
from fastapi.sse import EventSourceResponse, ServerSentEvent

example_event = ServerSentEvent(
    data={"step": 1, "text": "完成步骤 1"}, event="update", id="1", retry=1000
)

# 先观察事件对象的字段；实际传输格式将在下一节通过端口读取。
assert example_event.data["step"] == 1 and example_event.retry == 1000
print(example_event.model_dump(exclude_none=True))

{'data': {'step': 1, 'text': '完成步骤 1'}, 'event': 'update', 'id': '1', 'retry': 1000}


使用 response_class=EventSourceResponse，把路由写成生成器，每次 yield 一个 ServerSentEvent。下面固定生成三个事件，最后一个命名为 done，这是本示例的结束约定，不是 SSE 自动提供的结束类型。

为观察生成器退出，state 记录当前活动数 active、启动累计数 started、退出累计数 closed 和生成事件累计数 produced。它们只是本进程的实验计数；finally 在正常迭代结束或取消时减少活动数。

In [6]:
state = {"active": 0, "started": 0, "closed": 0, "produced": 0}

@app.get("/events", response_class=EventSourceResponse)
async def events() -> AsyncIterator[ServerSentEvent]:
    state["active"] += 1
    state["started"] += 1
    try:
        for step in range(1, 4):
            state["produced"] += 1
            yield ServerSentEvent(
                data={"step": step, "text": f"完成步骤 {step}"},
                event="done" if step == 3 else "update",
                id=str(step),
                retry=1000,
            )
            if step < 3:
                await asyncio.sleep(1.0)
    finally:
        state["active"] -= 1
        state["closed"] += 1
        print("SSE 生成器已退出", flush=True)

再提供一个只读状态接口。客户端需要读取服务进程中的计数，不能用 Notebook 里另一个 app 对象的 state 代替。

In [7]:
@app.get("/state")
def stream_state() -> dict[str, int]:
    return state.copy()

## 5 从真实响应读出事件字段

先用 HTTPX 查看 SSE 的原始文本行。以下辅助函数只收集本例一个事件到空行为止，忽略以冒号开头的注释行；它用于观察本例格式，不实现完整的浏览器 SSE 解析与重连逻辑。

In [8]:
async def read_event_lines(lines: AsyncIterator[str]) -> list[str]:
    fields = []
    async for line in lines:
        if line == "" and fields:
            return fields
        if line and not line.startswith(":"):
            fields.append(line)
    raise AssertionError("尚未读到完整事件就结束了响应")

使用同一个行迭代器读取三个事件，再确认响应结束。本例每个事件只有一行 data，可直接提取并解析它；一般 SSE 消息可以包含多行 data，浏览器会按标准拼接。

In [9]:
import json

async with httpx.AsyncClient(base_url=base_url, timeout=3.0, trust_env=False) as client:
    async with client.stream("GET", "/events") as response:
        response.raise_for_status()
        assert response.headers["content-type"].startswith("text/event-stream")
        lines = response.aiter_lines()
        blocks = [await read_event_lines(lines) for _ in range(3)]
        remaining = [line async for line in lines]
    after_complete = (await client.get("/state")).json()

for step, block in enumerate(blocks, start=1):
    fields = dict(line.split(": ", 1) for line in block)
    assert fields["id"] == str(step) and fields["retry"] == "1000"
    assert fields["event"] == ("done" if step == 3 else "update")
    assert json.loads(fields["data"])["step"] == step
    print("\n".join(block), end="\n\n")
assert not remaining and after_complete["active"] == 0
assert response.is_closed and client.is_closed
print("完成后活动生成器：", after_complete["active"])

event: update
data: {"step": 1, "text": "\u5b8c\u6210\u6b65\u9aa4 1"}
id: 1
retry: 1000

event: update
data: {"step": 2, "text": "\u5b8c\u6210\u6b65\u9aa4 2"}
id: 2
retry: 1000

event: done
data: {"step": 3, "text": "\u5b8c\u6210\u6b65\u9aa4 3"}
id: 3
retry: 1000

完成后活动生成器： 0


## 6 收到首个事件后断开

提前退出 HTTPX 的 stream 上下文会关闭该响应。服务端处理断开、生成器执行 finally 与客户端关闭发生在不同任务中，不能把刚关闭后的瞬间计数当作最终结果。

下面给状态检查一个 3 秒总时限，在看到 active 为 0 且 closed 增加后返回；每次查询之间让出执行机会。这是按清理条件等待，不是断言取消必须花某个固定时间。

In [10]:
async def wait_for_cleanup(client: httpx.AsyncClient, expected_closed: int) -> dict:
    async with asyncio.timeout(3.0):
        while True:
            response = await client.get("/state")
            response.raise_for_status()
            observed = response.json()
            if observed["active"] == 0 and observed["closed"] >= expected_closed:
                return observed
            await asyncio.sleep(0.05)

现在先读取服务计数，再打开事件流并完整读到第一个事件的空行，随后退出上下文。此实验期间只由本单元访问事件接口，避免把其他连接计入增量。

In [11]:
async with httpx.AsyncClient(base_url=base_url, timeout=3.0, trust_env=False) as client:
    before = (await client.get("/state")).json()
    async with client.stream("GET", "/events") as response:
        response.raise_for_status()
        lines = response.aiter_lines()
        first_event = await read_event_lines(lines)
        assert "id: 1" in first_event
    after = await wait_for_cleanup(client, before["closed"] + 1)

assert response.is_closed and client.is_closed
assert after["started"] == before["started"] + 1
assert after["produced"] - before["produced"] == 1
print("收到的事件：", first_event)
print("活动数：", after["active"], "退出数增量：", after["closed"] - before["closed"])
print("生成事件数增量：", after["produced"] - before["produced"])

收到的事件： ['event: update', 'data: {"step": 1, "text": "\\u5b8c\\u6210\\u6b65\\u9aa4 1"}', 'id: 1', 'retry: 1000']
活动数： 0 退出数增量： 1
生成事件数增量： 1


## 7 用浏览器 EventSource 消费并关闭

原生 EventSource 读取 SSE，会把命名事件交给对应的 addEventListener；事件对象的 data 是文本，lastEventId 是最近的事件标识。没有 event 字段的消息可用 message 监听器处理。本例监听 update 和 done。

事件数据先按本例结构解析，再写入页面的 textContent，把消息作为文本显示。下面是 events.html 中负责显示事件的部分；output 对应页面的 pre 元素。

```javascript
function appendEvent(event) {
  const payload = JSON.parse(event.data);
  if (!Number.isInteger(payload.step) || typeof payload.text !== 'string') {
    throw new Error('事件数据格式不正确');
  }
  output.textContent += `${event.type} #${event.lastEventId}: ${payload.text}\n`;
}
```

SSE 连接结束后，浏览器通常会尝试重连。有限任务必须有自己的结束约定：本例收到 done 后调用 close，并在“首个事件后关闭”模式中收到第一条 update 就调用 close。close 会停止当前连接及其重连，readyState 变成 CLOSED。

页面里的 finish 调用指定连接的 close，并更新状态文字。start 每次关闭上一个连接，再注册本次监听器；stopAfterFirst 是是否在首个事件后停止的布尔值。

```javascript
function start(stopAfterFirst) {
  source?.close();
  output.textContent = '';
  status.textContent = '正在连接';
  const connection = new EventSource('/events');
  source = connection;

  function receive(event) {
    try {
      appendEvent(event);
    } catch {
      finish(connection, '事件格式错误，已关闭连接');
      return;
    }
    if (event.type === 'done') {
      finish(connection, '3 个事件已接收，连接已关闭');
    } else if (stopAfterFirst) {
      finish(connection, '首个事件已接收，已提前关闭');
    }
  }

  connection.addEventListener('update', receive);
  connection.addEventListener('done', receive);
  connection.addEventListener('error', () => finish(connection, '连接出错，已停止接收'));
}
```

页面和 API 使用同一个服务。app.py 用 FileResponse 返回固定 HTML 文件，脚本中的页面路径相对于 app.py 所在目录，不由请求参数提供；HTML 内联少量原生 JavaScript。

In [12]:
from pathlib import Path

from fastapi.responses import FileResponse

page_path = Path("scripts/20-streaming-and-sse/events.html")


@app.get("/", include_in_schema=False)
def event_page() -> FileResponse:
    return FileResponse(page_path, media_type="text/html")


async with httpx.AsyncClient(base_url=base_url, timeout=3.0, trust_env=False) as client:
    page = await client.get("/")
    page.raise_for_status()
assert "new EventSource('/events')" in page.text
print(page.status_code, page.headers["content-type"])

200 text/html; charset=utf-8


Step 1：保持第 2 节的服务运行，在浏览器打开 http://127.0.0.1:8200/。

Step 2：点击“接收3个事件”，观察 update #1、update #2 和 done #3 依次出现，状态变成“3 个事件已接收，连接已关闭”。

![浏览器依次收到 update 1、update 2 和 done 3，并显示连接已关闭](image/20-sse-complete.png)

图中保留收到的三个事件；最后的 done 对应页面的完成状态和主动关闭。

Step 3：点击页面的“查看服务端状态”，确认 active 为 0；关闭状态页，回到事件页面。

Step 4：点击“首个事件后关闭”，观察页面只出现 update #1，状态变成“首个事件已接收，已提前关闭”；再次查看服务端状态，确认 active 为 0，closed 增加。

Step 5：需要手动停止时点击“停止接收”。页面离开时也会调用 close；实验页面的连接错误同样会停止接收，避免无穷重连。

Step 6：回到服务终端按 Ctrl+C，等待应用关闭完成；刷新首页，确认端口已不可访问。

本实验直接连接本地服务；生成、网络传输与浏览器呈现不是同一个时刻。读取结果以内容、事件顺序和清理状态为依据，不把固定到达间隔作为成功标准。

## 本章小结

（1）StreamingResponse 从生成器逐步发送内容，客户端读取块与 yield 边界不必一致；文本行和事件边界需要按格式识别。

（2）SSE 使用 UTF-8 与 text/event-stream，以空行分隔事件。FastAPI 原生 ServerSentEvent 的 data 会编码为 JSON。

（3）event、id、retry 各自表示事件类型、标识与重连等待；id 不自动实现重放，done 是本例约定。

（4）有限浏览器消费在正常结束或提前停止时调用 EventSource.close。客户端关闭后，还要观察服务端生成器退出，最后关闭独立服务。

自查：为什么不能按一次网络读取就解析一个完整 SSE 事件？为什么服务端生成完三个事件后，浏览器仍需要主动 close？

## 练习

1. 把文本流内容改为三行中文，保持 UTF-8 编码。用 aiter_lines 检查完整中文行；按字节读取时先拼接再解码，不假定一个字节块包含完整字符。
2. 把前两个 SSE 事件的类型改为 progress，同时修改浏览器监听器。确认依次显示 progress #1、progress #2、done #3，最终连接关闭。
3. 在第二个事件后提前关闭 HTTPX 响应，确认只读到 id 为 1、2 的事件，服务端 active 最终为 0，closed 增加一次。
4. 给 data 增加整数 percent 字段，分别发送 30、60、100。浏览器校验并展示进度值，接收完毕后仍关闭连接；核对原始 SSE 的 data 是单层 JSON 对象。

### 提示

- 第 1 题文本格式和网络分块是两层不同约定，编码后的字符可能跨读取块。
- 第 2 题服务端 event 名称与 addEventListener 的事件名必须一致。
- 第 3 题复用 read_event_lines 连续读取两次，退出 stream 后按状态条件等待。
- 第 4 题给 ServerSentEvent.data 传字典，页面通过 JSON.parse 解析后检查字段。

## 参考与引用来源

- **FastAPI 官方文档**：[StreamingResponse](https://fastapi.tiangolo.com/advanced/custom-response/#streamingresponse)，定位生成器响应、await 与取消机会；[Server-Sent Events](https://fastapi.tiangolo.com/tutorial/server-sent-events/)，定位 0.135.0 版本说明、Stream SSE with FastAPI、ServerSentEvent、Resuming with Last-Event-ID，支持原生响应类、字段、data 的 JSON 编码与续传职责；[FileResponse](https://fastapi.tiangolo.com/advanced/custom-response/#fileresponse)，支持固定 HTML 文件响应。
- **WHATWG HTML Living Standard**：[Server-sent events](https://html.spec.whatwg.org/multipage/server-sent-events.html)，定位 §9.2.2 EventSource interface、§9.2.3 Processing model、§9.2.4 Last-Event-ID、§9.2.5 Parsing an event stream、§9.2.6 Interpreting an event stream，支持 UTF-8、空行事件边界、字段含义、多行 data、命名事件、重连与 close。
- **WHATWG DOM Standard**：[Node.textContent](https://dom.spec.whatwg.org/#dom-node-textcontent)，定位 setter 创建文本内容，支持页面以文本展示事件；[EventTarget.addEventListener](https://dom.spec.whatwg.org/#dom-eventtarget-addeventlistener)，支持按事件类型注册监听器。
- **TC39 ECMAScript 规范**：[JSON.parse](https://tc39.es/ecma262/multipage/structured-data.html#sec-json.parse) 与 [Number.isInteger](https://tc39.es/ecma262/multipage/numbers-and-dates.html#sec-number.isinteger)，支持页面解析事件数据和检查整数步骤。
- **HTTPX 官方文档**：[Async Support](https://www.python-httpx.org/async/#streaming-responses)，定位 stream、aiter_bytes、aiter_lines 与响应关闭；[Developer Interface](https://www.python-httpx.org/api/#response)，定位响应迭代器和 chunk_size 参数；[QuickStart](https://www.python-httpx.org/quickstart/#streaming-responses)，定位按行读取及换行规范化；[Environment Variables](https://www.python-httpx.org/environment_variables/)，支持 trust_env=False。
- **Python 3.12 官方文档**：[Coroutines and Tasks](https://docs.python.org/3.12/library/asyncio-task.html)，定位 Task Cancellation、Sleeping 和 Timeouts，支持 await、finally 与有限清理等待；[异步生成器](https://docs.python.org/3.12/reference/expressions.html#asynchronous-generator-functions)，支持异步生成器的逐次恢复与退出；[json.loads](https://docs.python.org/3.12/library/json.html#json.loads)，支持解析本例的事件数据。
- **RFC Editor**：[RFC 9293 §2.2 与 §3.7](https://www.rfc-editor.org/rfc/rfc9293.html#section-3.7)，支持 TCP 字节流及分段边界与应用写入边界不保证一致。
- **Uvicorn 官方文档**：[Settings](https://uvicorn.dev/settings/)，定位 Application 与 Socket Binding，支持 app 导入字符串、本地地址和端口设置。